# E-Commerce Logistics and Delivery Analysis

**Professional Logistics Analytics & Machine Learning Notebook**

Explore delivery performance, logistics costs, operational drivers, customer satisfaction, and predictive modeling.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report

df=pd.read_csv('/kaggle/input/e-commerce-logistics-and-delivery-dataset/ecommerce_logistics_delivery.csv',parse_dates=['Order_Date'])
df.head()

## 1. Dataset Overview & Quality

In [ ]:
print('Rows:',f'{len(df):,}','| Columns:',len(df.columns))
print('Missing values:',df.isna().sum().sum(),'| Duplicates:',df.duplicated().sum())
display(df.describe(include='all').T)

## 2. Delivery Performance

In [ ]:
display(df['Delivery_Status'].value_counts(normalize=True).mul(100).round(2).rename('Percentage').to_frame())
perf=df.groupby('Shipping_Method').agg(Orders=('Order_ID','count'),Avg_Delivery_Hours=('Estimated_Delivery_Hours','mean'),Avg_Delay_Hours=('Delivery_Delay_Hours','mean'),On_Time_Rate=('On_Time_Delivery_Flag','mean'),Avg_Logistics_Cost=('Logistics_Cost','mean'))
perf['On_Time_Rate']*=100
display(perf.round(2))

In [ ]:
plt.figure(figsize=(8,5)); df.groupby('Shipping_Method')['Delivery_Delay_Hours'].mean().sort_values().plot(kind='bar'); plt.title('Average Delivery Delay by Shipping Method'); plt.ylabel('Hours'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Operational Drivers

In [ ]:
drivers=['Distance_KM','Warehouse_Processing_Hours','Traffic_Delay_Minutes','Weather_Delay_Minutes','Warehouse_Capacity_Utilization','Inventory_Availability','Courier_Rating','Courier_Experience_Years']
display(df[drivers+['Delivery_Delay_Hours']].corr(numeric_only=True)['Delivery_Delay_Hours'].sort_values(ascending=False).to_frame('Correlation'))

## 4. Delivery Delay Prediction

In [ ]:
target='Delivery_Delay_Hours'; drop=['Order_ID','Order_Date','Delivery_Status','On_Time_Delivery_Flag','Cancellation_Flag','Return_Flag']
X=df.drop(columns=[target]+drop); y=df[target]
cat=X.select_dtypes(include='object').columns.tolist(); num=X.select_dtypes(exclude='object').columns.tolist()
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
model=Pipeline([('pre',pre),('model',RandomForestRegressor(n_estimators=200,max_depth=18,min_samples_leaf=3,random_state=42,n_jobs=-1))])
model.fit(Xtr,ytr); pred=model.predict(Xte)
print('MAE:',round(mean_absolute_error(yte,pred),3)); print('RMSE:',round(mean_squared_error(yte,pred)**.5,3)); print('R2:',round(r2_score(yte,pred),3))

## 5. On-Time Delivery Classification

In [ ]:
target='On_Time_Delivery_Flag'; drop=['Order_ID','Order_Date','Delivery_Status','Delivery_Delay_Hours','Cancellation_Flag','Return_Flag']
X=df.drop(columns=[target]+drop); y=df[target]
cat=X.select_dtypes(include='object').columns.tolist(); num=X.select_dtypes(exclude='object').columns.tolist()
pre2=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
clf=Pipeline([('pre',pre2),('model',RandomForestClassifier(n_estimators=200,max_depth=18,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=-1))])
clf.fit(Xtr,ytr); pred=clf.predict(Xte)
print('Accuracy:',round(accuracy_score(yte,pred),3),'Precision:',round(precision_score(yte,pred,zero_division=0),3),'Recall:',round(recall_score(yte,pred,zero_division=0),3),'F1:',round(f1_score(yte,pred,zero_division=0),3))
print(classification_report(yte,pred,target_names=['Late','On Time'],zero_division=0))

## 6. Business Recommendations

- Compare delivery partners using on-time rate, delay hours, cost, and satisfaction.
- Monitor warehouse capacity and processing time to reduce fulfillment bottlenecks.
- Account for traffic and weather disruptions in delivery estimates.
- Use delay prediction for proactive customer communication.
- Optimize shipping method selection using both cost and service-level targets.

**Important:** This is synthetic data intended for education, research, portfolio projects, and ML experimentation.